<a href="https://colab.research.google.com/github/nreyes18/python-text-analytics-excel-automation/blob/main/Python_Text_Analytics_%26_Excel_Automation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:

# Project: Text File Parser to Excel Workbook
# Files needed: novels.txt and reviews.txt

!pip install openpyxl -q

import os
import re
import html
import json
from datetime import datetime

import openpyxl
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.worksheet.table import Table, TableStyleInfo
from openpyxl.utils import get_column_letter

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False


required_files = ["novels.txt", "reviews.txt"]

for file_name in required_files:
    if not os.path.exists(file_name):
        print(f"Please upload {file_name}")
        if IN_COLAB:
            files.upload()
        else:
            raise FileNotFoundError(f"{file_name} was not found.")

def count_words(text):
    """
    Counts words in a text string.
    """
    return len(re.findall(r"\b[\w'-]+\b", text))


def classify_sentiment(text):
    """
    Simple rule-based sentiment classifier.
    This does not use AI or an API.
    It counts positive and negative words and assigns a sentiment.
    """
    positive_words = {
        "great", "good", "excellent", "amazing", "love", "loved", "happy",
        "better", "best", "nice", "fun", "easy", "smooth", "clear",
        "durable", "fast", "comfortable", "convenient", "quality"
    }

    negative_words = {
        "bad", "poor", "slow", "issue", "issues", "problem", "problems",
        "regret", "harder", "lack", "never", "slowest", "connection"
    }

    clean_text = text.lower()
    words = re.findall(r"[a-zA-Z']+", clean_text)

    positive_score = sum(1 for word in words if word in positive_words)
    negative_score = sum(1 for word in words if word in negative_words)

    if positive_score > negative_score:
        return "Positive"
    elif negative_score > positive_score:
        return "Negative"
    else:
        return "Neutral"


def clean_text_spacing(text):
    """
    Removes extra spacing and line breaks.
    """
    return re.sub(r"\s+", " ", text).strip()

def read_reviews_to_list(file_path):
    """
    Reads reviews.txt into a list.
    Each list element is a dictionary with:
    Author, Review Title, Review Text, Product, Review Word Count, Review Sentiment.
    """

    with open(file_path, "r", encoding="utf-8") as file:
        raw_text = file.read().strip()

    review_blocks = re.split(r"\n\s*\n(?=Review\s+\d+:)", raw_text)

    reviews_list = []

    review_header_pattern = re.compile(r"^Review\s+(\d+):\s*(.+)$")
    review_info_pattern = re.compile(
        r"^(.*?)'s review of\s+<a\s+href=\".*?\">(.*?)</a>:\s*$",
        re.IGNORECASE
    )

    for block in review_blocks:
        lines = [line.strip() for line in block.splitlines() if line.strip()]

        if len(lines) < 3:
            continue

        header_match = review_header_pattern.match(lines[0])
        info_match = review_info_pattern.match(lines[1])

        if not header_match or not info_match:
            continue

        review_number = int(header_match.group(1))
        review_title = html.unescape(header_match.group(2).strip())

        review_author = html.unescape(info_match.group(1).strip())
        review_product = html.unescape(info_match.group(2).strip())

        review_text = clean_text_spacing(" ".join(lines[2:]))
        review_text = html.unescape(review_text)

        review_word_count = count_words(review_text)
        review_sentiment = classify_sentiment(review_text)

        review_dictionary = {
            "Review Number": review_number,
            "Author": review_author,
            "Review Title": review_title,
            "Review Text": review_text,
            "Product": review_product,
            "Review Word Count": review_word_count,
            "Review Sentiment": review_sentiment
        }

        reviews_list.append(review_dictionary)

    return reviews_list


def read_novels_to_dictionary(file_path):
    """
    Reads novels.txt into a dictionary.
    Author name is the key.
    The value is a list of dictionaries.
    Each dictionary includes Rank, Title, and Date.
    """

    novels_by_author = {}

    novel_pattern = re.compile(
        r"^\s*(\d+)\.\s+(.*?)\s*\((.*?),\s*([0-9]{4}(?:-[0-9]{4})?)\)\s*$"
    )

    with open(file_path, "r", encoding="utf-8") as file:
        for line in file:
            line = line.strip()

            if not line:
                continue

            match = novel_pattern.match(line)

            if match:
                rank = int(match.group(1))
                title = match.group(2).strip()
                author = match.group(3).strip()
                date = match.group(4).strip()

                novel_dictionary = {
                    "Rank": rank,
                    "Title": title,
                    "Date": date
                }

                if author not in novels_by_author:
                    novels_by_author[author] = []

                novels_by_author[author].append(novel_dictionary)

    return novels_by_author

def style_header_row(worksheet, row_number=1):
    """
    Styles the header row of a worksheet.
    """
    header_fill = PatternFill("solid", fgColor="1F4E78")
    header_font = Font(bold=True, color="FFFFFF")

    thin_border = Border(
        left=Side(style="thin", color="D9E2F3"),
        right=Side(style="thin", color="D9E2F3"),
        top=Side(style="thin", color="D9E2F3"),
        bottom=Side(style="thin", color="D9E2F3")
    )

    for cell in worksheet[row_number]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center", wrap_text=True)
        cell.border = thin_border


def auto_format_sheet(worksheet, max_width=45):
    """
    Applies basic professional formatting to a worksheet.
    """
    thin_border = Border(
        left=Side(style="thin", color="D9E2F3"),
        right=Side(style="thin", color="D9E2F3"),
        top=Side(style="thin", color="D9E2F3"),
        bottom=Side(style="thin", color="D9E2F3")
    )

    for row in worksheet.iter_rows():
        for cell in row:
            cell.alignment = Alignment(vertical="top", wrap_text=True)
            cell.border = thin_border

    for column_cells in worksheet.columns:
        column_letter = get_column_letter(column_cells[0].column)
        longest_value = 0

        for cell in column_cells:
            if cell.value is not None:
                longest_value = max(longest_value, len(str(cell.value)))

        adjusted_width = min(longest_value + 2, max_width)
        worksheet.column_dimensions[column_letter].width = adjusted_width

    worksheet.freeze_panes = "A2"


def add_excel_table(worksheet, table_name):
    """
    Adds an Excel table to a worksheet if there is data.
    """
    if worksheet.max_row > 1 and worksheet.max_column > 1:
        table_ref = f"A1:{get_column_letter(worksheet.max_column)}{worksheet.max_row}"

        table = Table(displayName=table_name, ref=table_ref)

        style = TableStyleInfo(
            name="TableStyleMedium2",
            showFirstColumn=False,
            showLastColumn=False,
            showRowStripes=True,
            showColumnStripes=False
        )

        table.tableStyleInfo = style
        worksheet.add_table(table)


def create_excel_file(reviews_list, novels_by_author):
    """
    Creates data.xlsx with:
    1. Novel Statistics
    2. Review Statistics
    3. Novels by Author
    4. Dictionary Preview
    """

    workbook = Workbook()

    novel_stats_ws = workbook.active
    novel_stats_ws.title = "Novel Statistics"

    novel_stats_ws["A1"] = "File Name"
    novel_stats_ws["B1"] = "novels.txt"

    novel_stats_ws["A2"] = "File Size"
    novel_stats_ws["B2"] = os.path.getsize("novels.txt")

    novel_stats_ws["A3"] = "Number of Authors"
    novel_stats_ws["B3"] = len(novels_by_author)

    novel_stats_ws["A4"] = "Total Novel Records"
    novel_stats_ws["B4"] = sum(len(novels) for novels in novels_by_author.values())

    novel_stats_ws["A5"] = "Workbook Created"
    novel_stats_ws["B5"] = datetime.now().strftime("%Y-%m-%d %I:%M %p")

    for row in range(1, 6):
        novel_stats_ws[f"A{row}"].font = Font(bold=True)
        novel_stats_ws[f"A{row}"].fill = PatternFill("solid", fgColor="D9EAF7")

    novel_stats_ws.column_dimensions["A"].width = 25
    novel_stats_ws.column_dimensions["B"].width = 35
    review_stats_ws = workbook.create_sheet("Review Statistics")

    review_headers = [
        "Review Number",
        "Review Title",
        "Review Author",
        "Review Product",
        "Review Text",
        "Review Length",
        "Review Word Count",
        "Average Word Length",
        "Review Sentiment"
    ]

    review_stats_ws.append(review_headers)

    for review in reviews_list:
        review_stats_ws.append([
            review["Review Number"],
            review["Review Title"],
            review["Author"],
            review["Product"],
            review["Review Text"],
            len(review["Review Text"]),
            review["Review Word Count"],
            "",
            review["Review Sentiment"]
        ])

    # Add Excel formula for average word length.
    for row_number in range(2, review_stats_ws.max_row + 1):
        review_stats_ws[f"H{row_number}"] = f"=IF(G{row_number}=0,0,F{row_number}/G{row_number})"

    style_header_row(review_stats_ws)
    auto_format_sheet(review_stats_ws)
    add_excel_table(review_stats_ws, "ReviewStatisticsTable")

    review_stats_ws.column_dimensions["E"].width = 70
    review_stats_ws.column_dimensions["H"].width = 20

    novels_ws = workbook.create_sheet("Novels by Author")

    novels_ws.append(["Author", "Rank", "Title", "Date"])

    for author, novels in novels_by_author.items():
        for novel in novels:
            novels_ws.append([
                author,
                novel["Rank"],
                novel["Title"],
                novel["Date"]
            ])

    style_header_row(novels_ws)
    auto_format_sheet(novels_ws)
    add_excel_table(novels_ws, "NovelsByAuthorTable")

    novels_ws.column_dimensions["A"].width = 28
    novels_ws.column_dimensions["C"].width = 45

    dictionary_ws = workbook.create_sheet("Dictionary Preview")

    dictionary_ws["A1"] = "Reviews List Preview"
    dictionary_ws["A2"] = json.dumps(reviews_list[:2], indent=4)

    dictionary_ws["A4"] = "Novels Dictionary Preview"
    first_three_authors = dict(list(novels_by_author.items())[:3])
    dictionary_ws["A5"] = json.dumps(first_three_authors, indent=4)

    dictionary_ws["A1"].font = Font(bold=True, size=14, color="FFFFFF")
    dictionary_ws["A1"].fill = PatternFill("solid", fgColor="1F4E78")
    dictionary_ws["A4"].font = Font(bold=True, size=14, color="FFFFFF")
    dictionary_ws["A4"].fill = PatternFill("solid", fgColor="1F4E78")

    dictionary_ws.column_dimensions["A"].width = 120
    dictionary_ws.row_dimensions[2].height = 220
    dictionary_ws.row_dimensions[5].height = 220

    dictionary_ws["A2"].alignment = Alignment(wrap_text=True, vertical="top")
    dictionary_ws["A5"].alignment = Alignment(wrap_text=True, vertical="top")

    # Save file
    output_file = "data.xlsx"
    workbook.save(output_file)

    return output_file


reviews_list = read_reviews_to_list("reviews.txt")
novels_by_author = read_novels_to_dictionary("novels.txt")

output_file = create_excel_file(reviews_list, novels_by_author)

print("Project completed successfully!")
print(f"Excel file created: {output_file}")
print(f"Total reviews processed: {len(reviews_list)}")
print(f"Total authors processed: {len(novels_by_author)}")
print(f"Total novels processed: {sum(len(novels) for novels in novels_by_author.values())}")

print("\nSample review dictionary:")
print(json.dumps(reviews_list[0], indent=4))

print("\nSample novels dictionary entry:")
first_author = list(novels_by_author.keys())[0]
print(first_author, ":", json.dumps(novels_by_author[first_author], indent=4))

if IN_COLAB:
    files.download(output_file)

Please upload novels.txt


Saving novels.txt to novels.txt
Please upload reviews.txt


Saving reviews.txt to reviews.txt
Project completed successfully!
Excel file created: data.xlsx
Total reviews processed: 6
Total authors processed: 71
Total novels processed: 100

Sample review dictionary:
{
    "Review Number": 1,
    "Author": "Cinder",
    "Review Title": "A great phone all around!",
    "Review Text": "I held on to my iPhone 6s as long as I could but the battery life was beginning to fade quickly so I finally decided to upgrade to the Xs. This phone is so fun and easy to use! I thought not having a home button would make it harder to control but everything is very smooth and easily accessible. The facial recognition is great and very smart, the camera is clear and takes great pictures, and the speaker has nice volume and great quality sound. The phone is also more durable than any other phone I've had and I finally feel comfortable using a light case with no screen protector instead of an otter box. I also love how long the battery lasts! I can use my phone all day

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>